# ORC (Optimized Row Columnar) Format Demonstration

## Overview
ORC is a columnar storage format designed for Hadoop and other big data frameworks. This notebook demonstrates ORC operations using **PySpark** for distributed data processing.

## Key Characteristics of ORC Format

### 1. **Columnar Storage**
   - Data is stored column-by-column instead of row-by-row
   - Benefits: Better compression, faster query execution on specific columns
   - Ideal for analytical queries that access only a subset of columns

### 2. **Compression**
   - Built-in compression support (ZLIB, SNAPPY, LZO)
   - Default is ZLIB with good compression ratios
   - Compression can reduce storage by 50-70% compared to CSV

### 3. **Encoding Techniques**
   - Run-length encoding (RLE)
   - Dictionary encoding for low-cardinality columns
   - Bit packing for numeric values
   - Patched Base for integers

### 4. **Stripe-based Structure**
   - File is divided into stripes (default 64MB)
   - Each stripe contains row data and index information
   - Allows parallel processing and partial reads

### 5. **Built-in Indexes**
   - Row group indexes for quick row lookups
   - Column indexes for predicate pushdown
   - Enables efficient filtering without reading all data

### 6. **Type Safety**
   - Schema information stored in file metadata
   - Type evolution support
   - No data type inference needed

### 7. **Performance Features**
   - Better performance for Hive and Spark
   - Vectorized read support
   - Efficient JOIN and aggregation operations
   - Reduces I/O and memory usage

### 8. **Use Cases**
   - Data warehousing
   - Log analysis
   - Time-series data
   - Historical data storage
   - ETL pipelines

In [2]:
# Set environment variables for PySpark
import os
import sys
try:
    import findspark
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "findspark"])
    import findspark

# Set Spark and Java homes using standard Homebrew paths
# For Spark 4.1.1 on Apple Silicon, the real home is in libexec
os.environ['SPARK_HOME'] = '/opt/homebrew/opt/apache-spark/libexec'
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['PYSPARK_PYTHON'] = sys.executable

# Initialize findspark
findspark.init(os.environ['SPARK_HOME'])

# Suppress verbose logging
import logging
logging.basicConfig(level=logging.ERROR)

# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType, TimestampType
import pyspark.sql.functions as F
from datetime import datetime, timedelta
import pandas as pd

print("Initializing PySpark...")

# Initialize SparkSession with master("local[*]") to prevent hanging
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ORC_Format_Demo") \
    .config("spark.sql.orc.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✓ SparkSession initialized successfully!")
print(f"Spark version: {spark.version}")
print("✓ ORC format support enabled")
print("✓ All operations will use PySpark")

Initializing PySpark...


KeyboardInterrupt: 

## Step 1: Create Sample Dataset (10 rows)

In [3]:
# Create sample data with 10 rows using PySpark
data = [
    (101, 'Alice Johnson', 'Sales', 65000, '2020-01-15', 3.85, True, 5),
    (102, 'Bob Smith', 'IT', 85000, '2020-02-15', 4.62, True, 8),
    (103, 'Charlie Brown', 'HR', 55000, '2020-03-15', 3.21, False, 3),
    (104, 'Diana Prince', 'Finance', 75000, '2020-04-15', 4.45, True, 6),
    (105, 'Eve Wilson', 'Sales', 68000, '2020-05-15', 3.95, True, 5),
    (106, 'Frank Miller', 'IT', 90000, '2020-06-15', 4.78, True, 10),
    (107, 'Grace Lee', 'HR', 58000, '2020-07-15', 3.12, False, 2),
    (108, 'Henry Davis', 'Finance', 78000, '2020-08-15', 4.35, True, 7),
    (109, 'Iris Anderson', 'Sales', 70000, '2020-09-15', 4.05, True, 6),
    (110, 'Jack Wilson', 'IT', 88000, '2020-10-15', 4.55, True, 9)
]

schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("employee_name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("hire_date", StringType(), False),
    StructField("performance_score", DoubleType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("bonus_percentage", IntegerType(), False)
])

# Create Spark DataFrame
df_spark = spark.createDataFrame(data, schema=schema)

# Convert hire_date to timestamp
df_spark = df_spark.withColumn("hire_date", F.to_timestamp("hire_date", "yyyy-MM-dd"))

print("Sample Dataset Created:")
print("=" * 80)
df_spark.show(10, truncate=False)
print("\n")
print(f"Dataset shape: ({df_spark.count()} rows, {len(df_spark.columns)} columns)")
print(f"\nData types:")
df_spark.printSchema()

NameError: name 'spark' is not defined

## Step 2: Write Data to ORC Format

**About ORC Writing with PySpark:**
- Spark's native ORC writer is highly optimized for distributed computing
- Automatically handles schema information and type safety
- Supports partitioning and compression out of the box
- Creates production-ready ORC files compatible with Hive, Spark, and other tools

In [ ]:
# Define the file path
orc_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees_orc'

# Write the DataFrame to ORC format
print("Writing data to ORC format...")
df_spark.coalesce(1).write \
    .mode("overwrite") \
    .format("orc") \
    .option("orc.compression", "SNAPPY") \
    .save(orc_file_path)

print(f"✓ ORC file created successfully at: {orc_file_path}")

# Get file size (ORC is stored as directory with multiple files)
total_size = 0
for root, dirs, files in os.walk(orc_file_path):
    for file in files:
        if file.endswith('.orc'):
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)

print(f"✓ File size: {total_size / 1024:.2f} KB")
print(f"✓ Compression: SNAPPY")
print(f"✓ Format: ORC (Columnar Storage)")

## Step 3: Read ORC File and Verify Data

**Reading ORC Files with PySpark:**
- Spark's ORC reader maintains schema information automatically
- Supports vectorized reads for better performance
- Automatic decompression and distributed processing
- Predicate pushdown optimization for selective column reads

In [ ]:
# Read the ORC file back
print("Reading ORC file...")
df_read_spark = spark.read.format("orc").load(orc_file_path)

print("Data read from ORC file:")
print("=" * 80)
df_read_spark.show(10, truncate=False)
print("\n")

# Verify data integrity
print("Data Integrity Check:")
print("=" * 80)
print(f"Original rows: {df_spark.count()}")
print(f"Read rows: {df_read_spark.count()}")
print(f"Data matches: {df_spark.count() == df_read_spark.count()}")

# Compare schemas
print("\nSchema comparison:")
print("-" * 80)
print("Original Schema:")
df_spark.printSchema()
print("\nRead Schema:")
df_read_spark.printSchema()

## Step 4: Test 1 - Schema and Metadata Inspection

In [ ]:
print("TEST 1: Schema and Metadata Inspection")
print("=" * 80)

# Get schema information
schema = df_read_spark.schema
print("\nSchema Information:")
print("-" * 80)
df_read_spark.printSchema()

# Get column information
print("\nColumn Details:")
print("-" * 80)
for i, field in enumerate(schema.fields):
    print(f"{i+1}. Column: {field.name}")
    print(f"   Type: {field.dataType}")
    print(f"   Nullable: {field.nullable}")
    print()

# Number of rows and columns
row_count = df_read_spark.count()
column_count = len(df_read_spark.columns)
print(f"Total rows: {row_count}")
print(f"Total columns: {column_count}")
print(f"Column names: {', '.join(df_read_spark.columns)}")

## Step 5: Test 2 - Column Selection (Predicate Pushdown)

**Theory:**
- One of ORC's key features is the ability to read only specific columns
- This reduces I/O and memory consumption for large files
- Spark's predicate pushdown filters data at the storage level, before reading
- Significant performance improvement for analytical queries on large datasets

In [ ]:
print("TEST 2: Column Selection and Predicate Pushdown")
print("=" * 80)

# Test 1: Read specific columns
selected_columns = ['employee_name', 'department', 'salary']
print(f"\nReading only columns: {selected_columns}")
print("-" * 80)

subset_df = df_read_spark.select(selected_columns)
subset_df.show(10, truncate=False)

row_count_subset = subset_df.count()
row_count_full = df_read_spark.count()
print(f"\nColumns selected: {len(selected_columns)} out of {len(df_read_spark.columns)}")
print(f"Rows: {row_count_subset}")

# Test 2: Read with filters (Predicate Pushdown)
print("\n" + "=" * 80)
print("Reading with filter (salary > 75000)")
print("-" * 80)

filtered_df = df_read_spark.filter(F.col('salary') > 75000)
filtered_df.show(10, truncate=False)

filtered_count = filtered_df.count()
original_count = df_read_spark.count()
print(f"\nFiltered rows: {filtered_count} out of {original_count}")
print(f"Reduction: {(1 - filtered_count / original_count) * 100:.1f}%")

## Step 6: Test 3 - Compression and Format Comparison

**Compression in ORC:**
- ORC supports multiple compression codecs: ZLIB, SNAPPY, LZO, ZSTD
- ZLIB provides better compression ratio but slower
- SNAPPY is faster with reasonable compression
- Spark's ORC writer uses SNAPPY by default for balanced performance

In [ ]:
print("TEST 3: Compression and Format Comparison")
print("=" * 80)

# Get ORC file size
orc_total_size = 0
for root, dirs, files in os.walk(orc_file_path):
    for file in files:
        if file.endswith('.orc'):
            file_path = os.path.join(root, file)
            orc_total_size += os.path.getsize(file_path)

print(f"\nORC Format (SNAPPY Compression):")
print(f"  File size: {orc_total_size / 1024:.2f} KB")

# Write as CSV for comparison
csv_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.csv'
df_read_spark.coalesce(1).write.mode("overwrite").format("csv").option("header", "true").save(csv_file_path)

csv_total_size = 0
for root, dirs, files in os.walk(csv_file_path):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            csv_total_size += os.path.getsize(file_path)

print(f"\nCSV Format (uncompressed):")
print(f"  File size: {csv_total_size / 1024:.2f} KB")

# Write as Parquet for comparison
parquet_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.parquet'
df_read_spark.coalesce(1).write.mode("overwrite").format("parquet").save(parquet_file_path)

parquet_total_size = 0
for root, dirs, files in os.walk(parquet_file_path):
    for file in files:
        if file.endswith('.parquet'):
            file_path = os.path.join(root, file)
            parquet_total_size += os.path.getsize(file_path)

print(f"\nParquet Format (SNAPPY Compression):")
print(f"  File size: {parquet_total_size / 1024:.2f} KB")

# Comparison summary
print("\n" + "=" * 80)
print("Format Comparison:")
print("-" * 80)
print(f"CSV (baseline):     {csv_total_size / 1024:.2f} KB (100%)")
print(f"ORC (SNAPPY):       {orc_total_size / 1024:.2f} KB ({orc_total_size / csv_total_size * 100:.1f}% of CSV)")
print(f"Parquet (SNAPPY):   {parquet_total_size / 1024:.2f} KB ({parquet_total_size / csv_total_size * 100:.1f}% of CSV)")

print("\n" + "=" * 80)
print("Space Saved vs CSV:")
print("-" * 80)
print(f"ORC:      Save {(1 - orc_total_size / csv_total_size) * 100:.1f}%")
print(f"Parquet:  Save {(1 - parquet_total_size / csv_total_size) * 100:.1f}%")

## Step 7: Test 4 - Data Statistics and Aggregations

**ORC Statistics:**
- ORC maintains min/max values for numeric columns in metadata
- Spark uses these statistics for query optimization
- Enables stripe pruning when filters are applied
- Dramatically improves performance for large datasets

In [ ]:
print("TEST 4: Data Statistics and Aggregations")
print("=" * 80)

# Perform various aggregations using Spark SQL
print("\nNumeric Column Statistics:")
print("-" * 80)

numeric_cols = ['salary', 'performance_score', 'bonus_percentage']
for col in numeric_cols:
    stats = df_read_spark.agg(
        F.count(col).alias(f"{col}_count"),
        F.mean(col).alias(f"{col}_mean"),
        F.min(col).alias(f"{col}_min"),
        F.max(col).alias(f"{col}_max"),
        F.stddev(col).alias(f"{col}_stddev")
    ).collect()[0]
    
    print(f"\n{col}:")
    print(f"  Count:   {int(stats[f'{col}_count'])}")
    print(f"  Mean:    {stats[f'{col}_mean']:.2f}")
    print(f"  Min:     {stats[f'{col}_min']}")
    print(f"  Max:     {stats[f'{col}_max']}")
    print(f"  Std Dev: {stats[f'{col}_stddev']:.2f}")

# Department-wise analysis
print("\n" + "=" * 80)
print("Department-wise Analysis:")
print("-" * 80)

dept_analysis = df_read_spark.groupBy('department').agg(
    F.count('*').alias('count'),
    F.mean('salary').alias('avg_salary'),
    F.min('salary').alias('min_salary'),
    F.max('salary').alias('max_salary'),
    F.mean('performance_score').alias('avg_performance')
).orderBy('department')

dept_analysis.show(10, truncate=False)

# Boolean column analysis
print("\n" + "=" * 80)
print("Boolean Column Analysis:")
print("-" * 80)

active_count = df_read_spark.filter(F.col('is_active') == True).count()
inactive_count = df_read_spark.filter(F.col('is_active') == False).count()
total_count = df_read_spark.count()

print(f"Active employees: {active_count}")
print(f"Inactive employees: {inactive_count}")
print(f"Activity rate: {active_count / total_count * 100:.1f}%")

## Step 8: Test 5 - Data Type Preservation

**Type Safety in ORC with PySpark:**
- ORC preserves exact data types when writing and reading
- No type inference needed (unlike CSV)
- Supports complex types: structs, arrays, maps
- Handles null values correctly throughout the pipeline

In [ ]:
print("TEST 5: Data Type Preservation")
print("=" * 80)

print("\nOriginal DataFrame Schema:")
print("-" * 80)
df_spark.printSchema()

print("\nTypes After ORC Read:")
print("-" * 80)
df_read_spark.printSchema()

print("\nDetailed Type Comparison:")
print("-" * 80)

original_fields = {field.name: str(field.dataType) for field in df_spark.schema.fields}
read_fields = {field.name: str(field.dataType) for field in df_read_spark.schema.fields}

type_comparison = []
for col in original_fields.keys():
    orig_type = original_fields[col]
    read_type = read_fields.get(col, "MISSING")
    matches = orig_type == read_type
    
    type_comparison.append({
        'Column': col,
        'Original': orig_type,
        'After ORC': read_type,
        'Match': '✓' if matches else '✗'
    })

type_df = pd.DataFrame(type_comparison)
print(type_df.to_string(index=False))

# Check for data loss
print("\n\nData Integrity Check:")
print("-" * 80)

all_match = True
for col in original_fields.keys():
    if col in read_fields:
        match = original_fields[col] == read_fields[col]
        status = "✓ OK" if match else "✗ MISMATCH"
    else:
        match = False
        status = "✗ MISSING"
    
    print(f"{col}: {status}")
    all_match = all_match and match

print(f"\nOverall Data Integrity: {'✓ PASSED' if all_match else '✗ FAILED'}")

## Step 9: Test 6 - Distributed Processing (Stripes)

**ORC Stripe Structure in Spark:**
- Data is organized into horizontal slices called stripes
- Default stripe size is 64MB
- Each stripe can be read independently across cluster nodes
- Enables efficient distributed processing and parallel reads
- Spark automatically schedules stripe reads to available executors

In [ ]:
print("TEST 6: Distributed Processing and Repartitioning")
print("=" * 80)

# Show partitions
print("\nPartition Information:")
print("-" * 80)
partition_count = df_read_spark.rdd.getNumPartitions()
print(f"Number of partitions: {partition_count}")
print(f"Rows per partition: {df_read_spark.count() // partition_count}")

# Repartition for batch processing
print("\nRepartitioning for batch processing:")
print("-" * 80)

df_repartitioned = df_read_spark.repartition(4)
partition_count_new = df_repartitioned.rdd.getNumPartitions()
print(f"New number of partitions: {partition_count_new}")

# Show data distribution across partitions
print("\nData distribution across partitions:")
for i in range(partition_count_new):
    partition_data = df_repartitioned.rdd.mapPartitionsWithIndex(
        lambda idx, rows: [(idx, len(list(rows)))]
    ).collect()
    if partition_data:
        idx, count = partition_data[0]
        if idx == i:
            print(f"  Partition {idx}: {count} rows")

print("\n" + "=" * 80)
print("ORC Stripe Information:")
print("-" * 80)
orc_total_size = 0
for root, dirs, files in os.walk(orc_file_path):
    for file in files:
        if file.endswith('.orc'):
            file_path = os.path.join(root, file)
            orc_total_size += os.path.getsize(file_path)
            print(f"  File: {file}, Size: {os.path.getsize(file_path) / 1024:.2f} KB")

print(f"\nTotal ORC file size: {orc_total_size / 1024:.2f} KB (< 64MB default stripe size)")
print(f"Fits in: 1 stripe")
print(f"\nFor larger files (e.g., 100GB):")
print(f"  - Would be split into multiple stripes (~64MB each)")
print(f"  - Each stripe could be processed on different nodes")
print(f"  - Enables true distributed processing across the cluster")

## Step 10: Test 7 - Performance Comparison

**ORC vs Other Formats with PySpark:**
- **ORC** vs CSV: Better compression, type safety, faster distributed reads
- **ORC** vs Parquet: Both columnar, ORC optimized for Hive, Parquet for Spark
- **ORC** vs JSON: Much better compression and performance for analytics
- **ORC** vs Avro: ORC is columnar (OLAP), Avro is row-based (OLTP)

In [ ]:
import time

print("TEST 7: Format Comparison and Performance")
print("=" * 80)

# Get sizes for all formats (using Spark)
csv_total_size = 0
for root, dirs, files in os.walk(csv_file_path):
    for file in files:
        if file.endswith('.csv'):
            csv_total_size += os.path.getsize(os.path.join(root, file))

parquet_total_size = 0
for root, dirs, files in os.walk(parquet_file_path):
    for file in files:
        if file.endswith('.parquet'):
            parquet_total_size += os.path.getsize(os.path.join(root, file))

orc_total_size = 0
for root, dirs, files in os.walk(orc_file_path):
    for file in files:
        if file.endswith('.orc'):
            orc_total_size += os.path.getsize(os.path.join(root, file))

# Write as JSON for comparison
json_file = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.json'
df_read_spark.coalesce(1).write.mode("overwrite").format("json").save(json_file)

json_total_size = 0
for root, dirs, files in os.walk(json_file):
    for file in files:
        if file.endswith('.json'):
            json_total_size += os.path.getsize(os.path.join(root, file))

# Build comparison table
comparison_data = {
    'Format': ['CSV', 'JSON', 'Parquet', 'ORC'],
    'File Size (KB)': [
        f"{csv_total_size / 1024:.2f}",
        f"{json_total_size / 1024:.2f}",
        f"{parquet_total_size / 1024:.2f}",
        f"{orc_total_size / 1024:.2f}"
    ],
    'Compression': [
        'None',
        'None',
        'SNAPPY',
        'SNAPPY'
    ],
    'Type Safety': [
        'No',
        'No',
        'Yes',
        'Yes'
    ],
    'Columnar': [
        'No',
        'No',
        'Yes',
        'Yes'
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nFormat Comparison (Using PySpark):")
print("-" * 80)
print(comparison_df.to_string(index=False))

# Calculate compression ratios
print("\n" + "=" * 80)
print("Compression Ratios (vs CSV baseline):")
print("-" * 80)

csv_baseline = csv_total_size
print(f"CSV:      {csv_baseline / 1024:.2f} KB (100%)")
print(f"JSON:     {json_total_size / 1024:.2f} KB ({json_total_size / csv_baseline * 100:.1f}%)")
print(f"Parquet:  {parquet_total_size / 1024:.2f} KB ({parquet_total_size / csv_baseline * 100:.1f}%)")
print(f"ORC:      {orc_total_size / 1024:.2f} KB ({orc_total_size / csv_baseline * 100:.1f}%)")

print("\n" + "=" * 80)
print("Space Saved vs CSV:")
print("-" * 80)
print(f"JSON:      Save {max(0, (1 - json_total_size / csv_baseline) * 100):.1f}%")
print(f"Parquet:   Save {(1 - parquet_total_size / csv_baseline) * 100:.1f}%")
print(f"ORC:       Save {(1 - orc_total_size / csv_baseline) * 100:.1f}%")

## Summary: ORC Format Benefits and Use Cases with PySpark

### Advantages of ORC Format:
1. **Excellent Compression** - Can achieve 50-70% reduction in file size
2. **Fast Query Performance** - Columnar format optimizes analytical queries
3. **Type Safety** - Schema is stored with data, no type inference needed
4. **Stripe-based Architecture** - Enables parallel processing and partial reads
5. **Built-in Indexing** - Row group indexes and column statistics for query optimization
6. **Predicate Pushdown** - Filters applied at storage level, not after reading
7. **Null Handling** - Efficiently handles null/missing values
8. **Standard Format** - Widely supported in Hadoop ecosystem, Hive, Spark
9. **PySpark Integration** - Native support in PySpark for distributed processing
10. **Partitioning Support** - Efficient storage and retrieval with partitioned datasets

### PySpark-Specific Advantages:
- **Distributed Writing** - Spark automatically parallelizes ORC writes across cluster
- **Distributed Reading** - Stripe-level parallelization across executor nodes
- **SQL Optimization** - Spark SQL automatically optimizes ORC queries
- **Cost Efficient** - Reduced storage and network I/O in cloud environments
- **Vectorized Operations** - Arrow-based columnar reads for faster processing

### When to Use ORC with PySpark:
- **Data Warehousing** - Large analytical datasets with PySpark queries
- **ETL Pipelines** - Data transformation and integration in Spark jobs
- **Hadoop/Spark Jobs** - Native support in Spark ecosystem
- **Long-term Storage** - Optimize disk space in data lakes
- **Distributed Analytics** - Large-scale data processing with Spark
- **Cost Optimization** - Reduce storage costs in cloud deployments

### When NOT to Use ORC:
- **Real-time Streaming** - Better for batch processing
- **Small Files** - CSV/JSON better for tiny datasets
- **Frequent Updates** - Delta Lake/Iceberg better for ACID
- **Single-machine Processing** - Use Pandas for local operations

### ORC File Structure:
```
┌─────────────────────────────────┐
│   Stripe 1 (e.g., 64MB)         │
│  ┌───────────────────────────┐  │
│  │ Column 1 Compressed Data  │  │
│  │ Column 2 Compressed Data  │  │
│  │ ...                       │  │
│  │ Index Data                │  │
│  │ Stripe Footer             │  │
│  └───────────────────────────┘  │
├─────────────────────────────────┤
│   Stripe 2                       │
│   ...                            │
├─────────────────────────────────┤
│   File Footer (Metadata)         │
│   File Postscript                │
└─────────────────────────────────┘
```

### Key Statistics:
- **Compression Ratio**: 50-70% reduction vs uncompressed
- **Read Performance**: 10-100x faster for selective column reads vs CSV
- **Data Integrity**: Perfect preservation of types and null values
- **Scalability**: Efficiently handles datasets from MB to TB+ scale
- **Distributed Processing**: Leverages Spark's distributed computing capabilities

# ORC (Optimized Row Columnar) Format Demonstration

## Overview
ORC is a columnar storage format designed for Hadoop and other big data frameworks. It provides efficient compression, encoding, and storage optimization for structured data.

## Key Characteristics of ORC Format

### 1. **Columnar Storage**
   - Data is stored column-by-column instead of row-by-row
   - Benefits: Better compression, faster query execution on specific columns
   - Ideal for analytical queries that access only a subset of columns

### 2. **Compression**
   - Built-in compression support (ZLIB, SNAPPY, LZO)
   - Default is ZLIB with good compression ratios
   - Compression can reduce storage by 50-70% compared to CSV

### 3. **Encoding Techniques**
   - Run-length encoding (RLE)
   - Dictionary encoding for low-cardinality columns
   - Bit packing for numeric values
   - Patched Base for integers

### 4. **Stripe-based Structure**
   - File is divided into stripes (default 64MB)
   - Each stripe contains row data and index information
   - Allows parallel processing and partial reads

### 5. **Built-in Indexes**
   - Row group indexes for quick row lookups
   - Column indexes for predicate pushdown
   - Enables efficient filtering without reading all data

### 6. **Type Safety**
   - Schema information stored in file metadata
   - Type evolution support
   - No data type inference needed

### 7. **Performance Features**
   - Better performance for Hive and Spark
   - Vectorized read support
   - Efficient JOIN and aggregation operations
   - Reduces I/O and memory usage

### 8. **Use Cases**
   - Data warehousing
   - Log analysis
   - Time-series data
   - Historical data storage
   - ETL pipelines

In [1]:
# Import required libraries
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
from datetime import datetime, timedelta
import numpy as np

print("All required libraries imported successfully!")
print(f"pandas version: {pd.__version__}")
print(f"pyarrow version: {pa.__version__}")
print("\nNote: We'll use PyArrow's ORC support for reading/writing ORC files")

All required libraries imported successfully!
pandas version: 2.3.3
pyarrow version: 22.0.0

Note: We'll use PyArrow's ORC support for reading/writing ORC files


## Step 1: Create Sample Dataset (10 rows)

In [2]:
# Create sample data with 10 rows
np.random.seed(42)

data = {
    'employee_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'employee_name': ['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Prince', 'Eve Wilson',
                      'Frank Miller', 'Grace Lee', 'Henry Davis', 'Iris Anderson', 'Jack Wilson'],
    'department': ['Sales', 'IT', 'HR', 'Finance', 'Sales', 'IT', 'HR', 'Finance', 'Sales', 'IT'],
    'salary': [65000, 85000, 55000, 75000, 68000, 90000, 58000, 78000, 70000, 88000],
    'hire_date': pd.date_range(start='2020-01-15', periods=10, freq='M'),
    'performance_score': np.random.uniform(3.0, 5.0, 10).round(2),
    'is_active': [True, True, False, True, True, True, False, True, True, True],
    'bonus_percentage': [5, 8, 3, 6, 5, 10, 2, 7, 6, 9]
}

# Create DataFrame
df = pd.DataFrame(data)

print("Sample Dataset Created:")
print("=" * 80)
print(df)
print("\n")
print(f"Dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

Sample Dataset Created:
   employee_id  employee_name department  salary  hire_date  \
0          101  Alice Johnson      Sales   65000 2020-01-31   
1          102      Bob Smith         IT   85000 2020-02-29   
2          103  Charlie Brown         HR   55000 2020-03-31   
3          104   Diana Prince    Finance   75000 2020-04-30   
4          105     Eve Wilson      Sales   68000 2020-05-31   
5          106   Frank Miller         IT   90000 2020-06-30   
6          107      Grace Lee         HR   58000 2020-07-31   
7          108    Henry Davis    Finance   78000 2020-08-31   
8          109  Iris Anderson      Sales   70000 2020-09-30   
9          110    Jack Wilson         IT   88000 2020-10-31   

   performance_score  is_active  bonus_percentage  
0               3.75       True                 5  
1               4.90       True                 8  
2               4.46      False                 3  
3               4.20       True                 6  
4               3.31  

/var/folders/_t/0rqjywnx1_z3rvn8xgy9yhg00000gn/T/ipykernel_91865/3535576610.py:10: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  'hire_date': pd.date_range(start='2020-01-15', periods=10, freq='M'),


## Step 2: Write Data to ORC Format

**About ORC Writing:**
- ORC files store data in a columnar format
- pyorc library provides a simple interface to write and read ORC files
- We'll write our DataFrame as an ORC file with compression enabled
- ORC files can store schema information, enabling type safety

In [5]:
# Convert pandas DataFrame to PyArrow Table (required for ORC)
table = pa.Table.from_pandas(df)

# Define the file path
orc_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.orc'

# Write the table to ORC format
print("Writing data to ORC format...")
import pyarrow.orc as orc
orc.write_table(table, orc_file_path)

print(f"✓ ORC file created successfully at: {orc_file_path}")
print(f"✓ File size: {os.path.getsize(orc_file_path) / 1024:.2f} KB")

# Display file information
file_stat = os.stat(orc_file_path)
print(f"✓ Created: {datetime.fromtimestamp(file_stat.st_ctime)}")
print(f"✓ Compression: Default (SNAPPY)")

Writing data to ORC format...
✓ ORC file created successfully at: /Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.orc
✓ File size: 1.42 KB
✓ Created: 2026-01-17 14:35:47.241872
✓ Compression: Default (SNAPPY)


## Step 3: Read ORC File and Verify Data

**Reading ORC Files:**
- PyArrow can efficiently read ORC files while maintaining schema information
- Supports column selection for optimized reads
- Automatic decompression of compressed data

In [6]:
# Read the ORC file back
print("Reading ORC file...")
import pyarrow.orc as orc
read_table = orc.read_table(orc_file_path)
df_read = read_table.to_pandas()

print("Data read from ORC file:")
print("=" * 80)
print(df_read)
print("\n")

# Verify data integrity
print("Data Integrity Check:")
print("=" * 80)
print(f"Original shape: {df.shape}")
print(f"Read shape: {df_read.shape}")
print(f"Data matches: {df.equals(df_read)}")

# Compare dtypes
print("\nData types comparison:")
print("-" * 80)
for col in df.columns:
    orig_type = df[col].dtype
    read_type = df_read[col].dtype
    match = "✓" if orig_type == read_type else "✗"
    print(f"{match} {col}: {orig_type} -> {read_type}")

Reading ORC file...
Data read from ORC file:
   employee_id  employee_name department  salary  hire_date  \
0          101  Alice Johnson      Sales   65000 2020-01-31   
1          102      Bob Smith         IT   85000 2020-02-29   
2          103  Charlie Brown         HR   55000 2020-03-31   
3          104   Diana Prince    Finance   75000 2020-04-30   
4          105     Eve Wilson      Sales   68000 2020-05-31   
5          106   Frank Miller         IT   90000 2020-06-30   
6          107      Grace Lee         HR   58000 2020-07-31   
7          108    Henry Davis    Finance   78000 2020-08-31   
8          109  Iris Anderson      Sales   70000 2020-09-30   
9          110    Jack Wilson         IT   88000 2020-10-31   

   performance_score  is_active  bonus_percentage  
0               3.75       True                 5  
1               4.90       True                 8  
2               4.46      False                 3  
3               4.20       True                 6  
4

## Step 4: Test 1 - Schema and Metadata Inspection

In [7]:
print("TEST 1: Schema and Metadata Inspection")
print("=" * 80)

# Get schema information
schema = read_table.schema
print("\nSchema Information:")
print("-" * 80)
print(schema)

# Get column information
print("\n\nColumn Details:")
print("-" * 80)
for i, field in enumerate(schema):
    print(f"{i+1}. Column: {field.name}")
    print(f"   Type: {field.type}")
    print(f"   Nullable: {field.nullable}")
    print()

# Number of rows and columns
print(f"Total rows: {read_table.num_rows}")
print(f"Total columns: {read_table.num_columns}")
print(f"Memory usage: {read_table.nbytes / 1024:.2f} KB")

TEST 1: Schema and Metadata Inspection

Schema Information:
--------------------------------------------------------------------------------
employee_id: int64
employee_name: string
department: string
salary: int64
hire_date: timestamp[ns]
performance_score: double
is_active: bool
bonus_percentage: int64


Column Details:
--------------------------------------------------------------------------------
1. Column: employee_id
   Type: int64
   Nullable: True

2. Column: employee_name
   Type: string
   Nullable: True

3. Column: department
   Type: string
   Nullable: True

4. Column: salary
   Type: int64
   Nullable: True

5. Column: hire_date
   Type: timestamp[ns]
   Nullable: True

6. Column: performance_score
   Type: double
   Nullable: True

7. Column: is_active
   Type: bool
   Nullable: True

8. Column: bonus_percentage
   Type: int64
   Nullable: True

Total rows: 10
Total columns: 8
Memory usage: 0.62 KB


## Step 5: Test 2 - Column Selection (Predicate Pushdown)

**Theory:**
- One of ORC's key features is the ability to read only specific columns
- This reduces I/O and memory consumption for large files
- Predicate pushdown filters data at the storage level, not after reading

In [8]:
print("TEST 2: Column Selection and Predicate Pushdown")
print("=" * 80)

# Test 1: Read specific columns
selected_columns = ['employee_name', 'department', 'salary']
print(f"\nReading only columns: {selected_columns}")
print("-" * 80)

subset_table = orc.read_table(orc_file_path, columns=selected_columns)
subset_df = subset_table.to_pandas()

print(subset_df)
print(f"\nMemory usage: {subset_table.nbytes / 1024:.2f} KB (vs {read_table.nbytes / 1024:.2f} KB full)")
print(f"Reduction: {((read_table.nbytes - subset_table.nbytes) / read_table.nbytes * 100):.1f}%")

# Test 2: Read with filters
print("\n" + "=" * 80)
print("Reading with filter (salary > 75000)")
print("-" * 80)

import pyarrow.compute as pc

filtered_table = read_table.filter(pc.field('salary') > 75000)
filtered_df = filtered_table.to_pandas()

print(filtered_df)
print(f"\nFiltered rows: {filtered_table.num_rows} out of {read_table.num_rows}")

TEST 2: Column Selection and Predicate Pushdown

Reading only columns: ['employee_name', 'department', 'salary']
--------------------------------------------------------------------------------
   employee_name department  salary
0  Alice Johnson      Sales   65000
1      Bob Smith         IT   85000
2  Charlie Brown         HR   55000
3   Diana Prince    Finance   75000
4     Eve Wilson      Sales   68000
5   Frank Miller         IT   90000
6      Grace Lee         HR   58000
7    Henry Davis    Finance   78000
8  Iris Anderson      Sales   70000
9    Jack Wilson         IT   88000

Memory usage: 0.30 KB (vs 0.62 KB full)
Reduction: 50.8%

Reading with filter (salary > 75000)
--------------------------------------------------------------------------------
   employee_id employee_name department  salary  hire_date  performance_score  \
0          102     Bob Smith         IT   85000 2020-02-29               4.90   
1          106  Frank Miller         IT   90000 2020-06-30             

## Step 6: Test 3 - Compression Comparison

**Compression in ORC:**
- ORC supports multiple compression codecs: ZLIB, SNAPPY, LZO
- ZLIB provides better compression ratio but slower
- SNAPPY is faster with reasonable compression
- Default compression reduces file size significantly

In [ ]:
print("TEST 3: Compression Comparison")
print("=" * 80)

# For ORC files with different compression (Note: pyarrow defaults to SNAPPY)
# Let's compare with CSV and JSON formats instead
compression_results = {}

# Current ORC file
orc_size = os.path.getsize(orc_file_path)
compression_results['orc_snappy'] = orc_size

# Compare with original CSV (convert to CSV and check)
csv_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.csv'
df.to_csv(csv_file_path, index=False)
csv_size = os.path.getsize(csv_file_path)

print(f"\nORC Format (SNAPPY Compression):")
print(f"  File size: {orc_size / 1024:.2f} KB")

print(f"\nCSV Format (uncompressed):")
print(f"  File size: {csv_size / 1024:.2f} KB")

# Create Parquet format for comparison
parquet_file_path = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.parquet'
pq.write_table(table, parquet_file_path)
parquet_size = os.path.getsize(parquet_file_path)

print(f"\nParquet Format (SNAPPY Compression):")
print(f"  File size: {parquet_size / 1024:.2f} KB")

# Comparison summary
print("\n" + "=" * 80)
print("Compression Summary:")
print("-" * 80)
print(f"CSV (baseline):     {csv_size / 1024:.2f} KB (100%)")
print(f"ORC (SNAPPY):       {orc_size / 1024:.2f} KB ({orc_size / csv_size * 100:.1f}% of CSV)")
print(f"Parquet (SNAPPY):   {parquet_size / 1024:.2f} KB ({parquet_size / csv_size * 100:.1f}% of CSV)")

print("\n" + "=" * 80)
print("Space Saved vs CSV:")
print("-" * 80)
print(f"ORC:      Save {(1 - orc_size / csv_size) * 100:.1f}%")
print(f"Parquet:  Save {(1 - parquet_size / csv_size) * 100:.1f}%")

## Step 7: Test 4 - Data Statistics and Aggregations

**ORC Statistics:**
- ORC maintains min/max values for numeric columns
- Helps in query optimization
- Enables pruning of data stripes that don't match query predicates

In [ ]:
print("TEST 4: Data Statistics and Aggregations")
print("=" * 80)

# Perform various aggregations
print("\nNumeric Column Statistics:")
print("-" * 80)

numeric_cols = ['salary', 'performance_score', 'bonus_percentage']
for col in numeric_cols:
    col_data = df_read[col]
    print(f"\n{col}:")
    print(f"  Count:   {col_data.count()}")
    print(f"  Mean:    {col_data.mean():.2f}")
    print(f"  Median:  {col_data.median():.2f}")
    print(f"  Min:     {col_data.min()}")
    print(f"  Max:     {col_data.max()}")
    print(f"  Std Dev: {col_data.std():.2f}")

# Department-wise analysis
print("\n" + "=" * 80)
print("Department-wise Analysis:")
print("-" * 80)

dept_analysis = df_read.groupby('department').agg({
    'salary': ['count', 'mean', 'min', 'max'],
    'performance_score': 'mean'
}).round(2)

print(dept_analysis)

# Boolean column analysis
print("\n" + "=" * 80)
print("Boolean Column Analysis:")
print("-" * 80)

active_count = df_read['is_active'].sum()
inactive_count = (~df_read['is_active']).sum()
print(f"Active employees: {active_count}")
print(f"Inactive employees: {inactive_count}")
print(f"Activity rate: {active_count / len(df_read) * 100:.1f}%")

## Step 8: Test 5 - Data Type Preservation

**Type Safety in ORC:**
- ORC preserves exact data types when writing and reading
- No type inference needed (unlike CSV)
- Supports complex types: structs, lists, maps
- Handles null values correctly

In [1]:
print("TEST 5: Data Type Preservation")
print("=" * 80)

print("\nOriginal DataFrame Types:")
print("-" * 80)
print(df.dtypes)

print("\n\nTypes After ORC Read:")
print("-" * 80)
print(df_read.dtypes)

print("\n\nDetailed Type Comparison:")
print("-" * 80)

type_comparison = []
for col in df.columns:
    orig_type = str(df[col].dtype)
    read_type = str(df_read[col].dtype)
    matches = orig_type == read_type
    
    type_comparison.append({
        'Column': col,
        'Original': orig_type,
        'After ORC': read_type,
        'Match': '✓' if matches else '✗'
    })

type_df = pd.DataFrame(type_comparison)
print(type_df.to_string(index=False))

# Check for data loss
print("\n\nData Integrity Check:")
print("-" * 80)

all_match = True
for col in df.columns:
    if df[col].dtype == object and df_read[col].dtype == object:
        # For object types, check if all values match
        match = (df[col] == df_read[col]).all()
    elif df[col].dtype == 'datetime64[ns]' and df_read[col].dtype == 'datetime64[ns]':
        # For datetime, check if all values match
        match = (df[col] == df_read[col]).all()
    else:
        # For numeric types
        match = np.allclose(df[col].fillna(0), df_read[col].fillna(0))
    
    status = "✓ OK" if match else "✗ MISMATCH"
    print(f"{col}: {status}")
    all_match = all_match and match

print(f"\nOverall Data Integrity: {'✓ PASSED' if all_match else '✗ FAILED'}")

TEST 5: Data Type Preservation

Original DataFrame Types:
--------------------------------------------------------------------------------


NameError: name 'df' is not defined

## Step 9: Test 6 - Batch Reading (Stripes)

**ORC Stripe Structure:**
- Data is organized into horizontal slices called stripes
- Default stripe size is 64MB
- Each stripe contains row data, index data, and metadata
- Enables efficient partial reads and parallel processing
- Multiple readers can process different stripes simultaneously

In [ ]:
print("TEST 6: Batch Reading Simulation")
print("=" * 80)

# Read file in batches
print("\nReading ORC file in batches (simulated):")
print("-" * 80)

batch_size = 3
total_rows = len(df_read)
num_batches = (total_rows + batch_size - 1) // batch_size

print(f"Total rows: {total_rows}")
print(f"Batch size: {batch_size}")
print(f"Number of batches: {num_batches}")

for batch_num in range(num_batches):
    start_idx = batch_num * batch_size
    end_idx = min(start_idx + batch_size, total_rows)
    batch_data = df_read.iloc[start_idx:end_idx]
    
    print(f"\nBatch {batch_num + 1}:")
    print(f"  Rows: {start_idx + 1} - {end_idx}")
    print(f"  Shape: {batch_data.shape}")
    print(f"  Employee IDs: {batch_data['employee_id'].tolist()}")

print("\n" + "=" * 80)
print("Stripe Information (For our small file):")
print("-" * 80)
print(f"In our 10-row example, the entire file fits in one stripe")
print(f"Stripe size would be: {orc_original_size / 1024:.2f} KB (< 64MB default)")
print(f"\nFor larger files (e.g., 100GB):")
print(f"  - Would be split into multiple stripes")
print(f"  - Each stripe could be processed independently")
print(f"  - Enables parallel processing across distributed systems")

## Step 10: Test 7 - Performance Comparison

**ORC vs Other Formats:**
- **ORC** vs CSV: Better compression, type safety, faster reads
- **ORC** vs Parquet: Both columnar, ORC optimized for Hive, Parquet for Spark
- **ORC** vs JSON: Much better compression and performance
- **ORC** vs Avro: Row-based, different use cases

In [ ]:
import time
import json

print("TEST 7: Format Comparison and Performance")
print("=" * 80)

# CSV (already created)
csv_size = os.path.getsize(csv_file_path)

# JSON format
json_file = '/Users/mukesh/Desktop/Trainings/nodeB/dataeng/jan_2026/File_Formats/employees.json'
start_time = time.time()
df.to_json(json_file, orient='records', indent=2)
json_write_time = time.time() - start_time
json_size = os.path.getsize(json_file)

# Parquet (already created)
parquet_size = os.path.getsize(parquet_file_path)

# ORC
orc_size = os.path.getsize(orc_file_path)

# Build comparison table
comparison_data = {
    'Format': ['CSV', 'JSON', 'Parquet', 'ORC'],
    'File Size (KB)': [
        f"{csv_size / 1024:.2f}",
        f"{json_size / 1024:.2f}",
        f"{parquet_size / 1024:.2f}",
        f"{orc_size / 1024:.2f}"
    ],
    'Compression': [
        'None',
        'None',
        'SNAPPY',
        'SNAPPY'
    ],
    'Type Safety': [
        'No (inference)',
        'No (inference)',
        'Yes',
        'Yes'
    ],
    'Read Speed': [
        'Slow',
        'Slow',
        'Fast',
        'Very Fast'
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nFormat Comparison:")
print("-" * 80)
print(comparison_df.to_string(index=False))

# Calculate compression ratios
print("\n" + "=" * 80)
print("Compression Ratios (vs CSV baseline):")
print("-" * 80)

csv_baseline = csv_size
print(f"CSV:      {csv_baseline / 1024:.2f} KB (100%)")
print(f"JSON:     {json_size / 1024:.2f} KB ({json_size / csv_baseline * 100:.1f}%)")
print(f"Parquet:  {parquet_size / 1024:.2f} KB ({parquet_size / csv_baseline * 100:.1f}%)")
print(f"ORC:      {orc_size / 1024:.2f} KB ({orc_size / csv_baseline * 100:.1f}%)")

print("\n" + "=" * 80)
print("Space Saved:")
print("-" * 80)
print(f"JSON vs CSV:      Save {max(0, (1 - json_size / csv_baseline) * 100):.1f}%")
print(f"Parquet vs CSV:   Save {(1 - parquet_size / csv_baseline) * 100:.1f}%")
print(f"ORC vs CSV:       Save {(1 - orc_size / csv_baseline) * 100:.1f}%")

## Summary: ORC Format Benefits and Use Cases

### Advantages of ORC Format:
1. **Excellent Compression** - Can achieve 50-70% reduction in file size
2. **Fast Query Performance** - Columnar format optimizes analytical queries
3. **Type Safety** - Schema is stored with data, no type inference needed
4. **Stripe-based Architecture** - Enables parallel processing and partial reads
5. **Built-in Indexing** - Row group indexes and column statistics for query optimization
6. **Predicate Pushdown** - Filters applied at storage level, not after reading
7. **Null Handling** - Efficiently handles null/missing values
8. **Standard Format** - Widely supported in Hadoop ecosystem, Hive, Spark

### When to Use ORC:
- **Data Warehousing** - Large analytical datasets
- **Log Analysis** - Time-series and event data
- **ETL Pipelines** - Data transformation and integration
- **Hadoop/Spark Jobs** - Native support in ecosystem
- **Long-term Storage** - Optimize disk space and I/O

### When NOT to Use ORC:
- **Real-time Streaming** - Better for batch processing
- **Small Files** - CSV/JSON better for tiny datasets
- **Frequent Updates** - Parquet or Delta Lake better for ACID
- **Cross-platform** - If working outside Hadoop ecosystem

### ORC File Structure:
```
┌─────────────────────────────────┐
│   Stripe 1 (e.g., 64MB)         │
│  ┌───────────────────────────┐  │
│  │ Column 1 Compressed Data  │  │
│  │ Column 2 Compressed Data  │  │
│  │ ...                       │  │
│  │ Index Data                │  │
│  │ Stripe Footer             │  │
│  └───────────────────────────┘  │
├─────────────────────────────────┤
│   Stripe 2                       │
│   ...                            │
├─────────────────────────────────┤
│   File Footer (Metadata)         │
│   File Postscript                │
└─────────────────────────────────┘
```

### Key Statistics:
- **Compression Ratio**: 50-70% reduction vs uncompressed
- **Read Performance**: 10-100x faster for selective column reads vs CSV
- **Data Integrity**: Perfect preservation of types and null values
- **Scalability**: Efficiently handles datasets from MB to TB+ scale